## Variational Quantum Eigensolver (VQE) for HeH+ Ion

### Description

This Jupyter notebook implements the Variational Quantum Eigensolver (VQE) algorithm to estimate the ground state energy of the HeH+ ion using quantum computing. The implementation uses PennyLane, a quantum machine learning library, to perform quantum chemistry calculations.

The notebook constructs the molecular Hamiltonian for HeH+ with a bond length of approximately 1.46 Å, and uses a simple ansatz consisting of an initial basis state preparation and a double excitation gate to optimize the energy.

In [1]:
%pip install pennylane

  Using cached pennylane-0.44.1-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached rustworkx-0.17.1-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached autograd-1.8.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached autoray-0.8.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached cachetools-7.0.5-py3-none-any.whl.metadata (5.6 kB)
  Using cached tomlkit-0.14.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached diastatic_malt-2.15.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached scipy_openblas32-0.3.31.188.0-py3-none-win_amd64.whl.metadata (57 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metada

In [36]:
import pennylane as qml
from pennylane import numpy as np

In [37]:
symbols = ["He", "H"]

geometry = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.4626278]], requires_grad=False) 

H, qubits = qml.qchem.molecular_hamiltonian(symbols, geometry, charge=1)


In [38]:
print(f"Number of qubits required: {qubits}")

Number of qubits required: 4


In [39]:
dev = qml.device("default.qubit", wires=qubits)

In [51]:
@qml.qnode(dev)
def ansatz(params):
    qml.BasisState(np.array([1, 1, 0, 0]), wires=range(qubits))
    qml.DoubleExcitation(params[0], wires=range(qubits))
    return qml.expval(H)

In [52]:
opt = qml.GradientDescentOptimizer(stepsize=0.4)
theta = np.array([0.0], requires_grad=True)

In [53]:
energy = [ansatz(theta)]

In [54]:
max_iterations = 40
convergence_tol = 1e-8

In [55]:
print("\nStarting VQE Optimization...")

for n in range(max_iterations):
    theta, prev_energy = opt.step_and_cost(ansatz, theta)
    energy.append(ansatz(theta))
    conv = np.abs(energy[-1] - prev_energy)
    if n % 2 == 0 :
     print(f"Iteration = {n}, Energy = {energy[-1]:.8f} Ha, Convergence = {conv:.8f}")
    if conv <= convergence_tol:
        break

print(f"\nVQE Optimization completed in {n} iterations.")
print(f"Final VQE Energy: {energy[-1]:.8f} Ha")
print(f"Optimal theta: {theta.item():.4f} radians")


Starting VQE Optimization...
Iteration = 0, Energy = -2.84831527 Ha, Convergence = 0.00653813
Iteration = 2, Energy = -2.85083019 Ha, Convergence = 0.00057356
Iteration = 4, Energy = -2.85104928 Ha, Convergence = 0.00004989
Iteration = 6, Energy = -2.85106833 Ha, Convergence = 0.00000434
Iteration = 8, Energy = -2.85106999 Ha, Convergence = 0.00000038
Iteration = 10, Energy = -2.85107013 Ha, Convergence = 0.00000003

VQE Optimization completed in 11 iterations.
Final VQE Energy: -2.85107014 Ha
Optimal theta: 0.1275 radians
